In [11]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
import sys
from pathlib import Path
from openai import OpenAI


import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))
from WebScrapper.scrape_site import scrape_site


In [12]:
load_dotenv(override=True)
api_key = os.getenv("GEMMA_API_KEY")

if api_key and len(api_key) > 10:
    print("API Key loaded successfully.")
else:
    print("Failed to load API Key. Please check your .env file.")

MODEL = "gemma3:latest"
GEMMA_BASE_URL = os.getenv("GEMMA_BASE_URL")

gemma = OpenAI(base_url=GEMMA_BASE_URL, api_key=api_key)

API Key loaded successfully.


In [13]:
link_system_prompt = """
You are provided with a list of website links and the contents of those web pages. 
You are able to decide which links are relevant and must include in the creative brochure, such as About page or Company Page etc.

You should respond in JSON as in this example:

{
  "start_url": "https://www.mcdonalds.com/ca/en-ca.html",
  "pages_scraped": 15,
  "pages": [
    {
      "title": "Title of the page",
      "text": "Contents of the page",
      "url": "https://url.com"
    },

"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links and contents on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email and Discord links.

Links (some might be relative links):

"""
    links = scrape_site(url)
    #print(links)
    user_prompt += "\n".join(links)
    return user_prompt

In [20]:
def select_relevant_links(url):
    response = gemma.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )

    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [21]:
select_relevant_links("https://www.mcdonalds.com/ca/en-ca.html")

{'start_url': 'https://www.mcdonalds.com/ca/en-ca.html', 'pages_scraped': 15, 'pages': [{'title': "Your Favourite Burgers, Fries & More | McDonald's Canada", 'text': "Skip To Main Content Language English Francais More McD's Find Jobs Franchising Menu View Full Menu Breakfast Beef Chicken Sandwiches & Wraps Snacks & Sides Desserts & Shakes Beverages McCafé McValue Happy Meal View Full Menu Nutrition Promotions McCafé Family MyMcDonald's Rewards Our Purpose and Impact Search Pick a Location X Close Menu Language English Francais More McD's Find Jobs Franchising Search Pick a Location Change Menu View Full Menu Breakfast Beef Chicken Sandwiches & Wraps Snacks & Sides Desserts & Shakes Beverages McCafé McValue Happy Meal View Full Menu Nutrition Promotions McCafé Family MyMcDonald's Rewards Our Purpose and Impact Good eats always deliver Enjoy $0 delivery fee* on the McD’s app.\u200b *11% service fee & $2 small order fee still apply.\u200b * Order Now * At participating McDonald’s in Cana

{'start_url': 'https://www.mcdonalds.com/ca/en-ca.html',
 'pages_scraped': 15,
 'pages': [{'title': "McDonald's",
   'text': "McDonald's is a global restaurant chain that serves a wide variety of food items, including hamburgers, french fries, chicken, salads, and desserts.  We are committed to providing our customers with a great taste experience and friendly service.",
   'url': 'https://www.mcdonalds.com/ca/en-ca.html'},
  {'title': 'Menu',
   'text': 'Explore our full menu of delicious items, from classic burgers and fries to breakfast, salads, and desserts.  You can also customize your order to create your perfect meal.',
   'url': 'https://www.mcdonalds.com/ca/en-ca/menu'},
  {'title': "McDonald's Canada",
   'text': "McDonald's has been a part of the Canadian landscape for over 50 years. We're proud to be a part of the communities we serve and committed to supporting local initiatives.",
   'url': 'https://www.mcdonalds.com/ca/en-ca/about-mcdonalds'},
  {'title': "McDonald's Car